In [0]:
%pip install mlflow databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
# Configuration from the embeddings notebook
VECTOR_SEARCH_ENDPOINT = "vector_search_facilities_endpoint"
INDEX_NAME = "workspace.default.facility_documents_index"
DOCS_TABLE = "workspace.default.facility_documents"

# Model serving configuration
CATALOG = "workspace"
SCHEMA = "default"
MODEL_NAME = "facility_rag_chatbot"
SERVING_ENDPOINT_NAME = "facility-rag-endpoint"

# RAG parameters
NUM_RESULTS = 3  # Number of documents to retrieve
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"  # Foundation model

In [0]:
import mlflow
import pandas as pd
from typing import Dict, Any, List
from databricks.vector_search.client import VectorSearchClient
from mlflow.models.signature import infer_signature
from mlflow.pyfunc import PythonModel

In [0]:
from dataclasses import dataclass
from typing import List, Optional
import json

@dataclass
class FacilityCitation:
    """One cited facility in a response."""
    facility_id  : str
    name         : str
    city         : str
    region       : str
    facility_type: str
    lat          : Optional[float]
    lon          : Optional[float]
    relevance_score: float   # from vector search — how relevant this facility was

def build_citations(context_docs: List[Dict]) -> List[Dict]:
    """
    Build structured citation objects from retrieved docs.
    These are returned with EVERY response so frontend always knows
    which facilities were used to generate the answer.
    """
    citations = []
    for rank, doc in enumerate(context_docs, 1):
        citation = FacilityCitation(
            facility_id    = str(doc["id"]),
            name           = doc["name"],
            city           = doc["city"],
            region         = doc["region"],
            facility_type  = doc["facility_type"],
            lat            = float(doc["lat"])   if doc["lat"]   else None,
            lon            = float(doc["lon"])   if doc["lon"]   else None,
            relevance_score= round(float(doc.get("score", 0)), 4)
        )
        citations.append({
            **citation.__dict__,
            "rank"         : rank,                    # position in search results
            "has_location" : citation.lat is not None and citation.lon is not None
        })
    return citations

def is_facility_response(answer: str, citations: List[Dict]) -> bool:
    """
    Returns True if the response is about specific facilities
    and the map button should be shown.

    Logic:
    - At least one citation has lat/lon
    - The answer actually mentions a facility name (not a generic response)
    - Not a "no results found" type response
    """
    # No mappable facilities found
    mappable = [c for c in citations if c["has_location"]]
    if not mappable:
        return False

    # Generic "not found" responses — don't show map button
    no_result_phrases = [
        "no matching facilities",
        "no facilities found",
        "cannot find",
        "not found in",
        "no information available",
        "context doesn't contain"
    ]
    answer_lower = answer.lower()
    if any(phrase in answer_lower for phrase in no_result_phrases):
        return False

    # Check that at least one facility name is actually mentioned in the answer
    facility_names_mentioned = any(
        c["name"].lower() in answer_lower
        for c in citations
    )

    return facility_names_mentioned

In [0]:
from mlflow.pyfunc import PythonModel
import pandas as pd
from typing import List, Dict, Any

class FacilityRAGModel(PythonModel):
    """RAG model for Virtue Foundation medical facility chatbot."""

    def load_context(self, context):
        """Initialize vector search client and load model artifacts."""
        from databricks.vector_search.client import VectorSearchClient
        
        def read_artifact(key):
            with open(context.artifacts[key], "r") as f:
                return f.read().strip()

        self.vector_search_endpoint = read_artifact("vector_search_endpoint")
        self.index_name             = read_artifact("index_name")
        self.llm_endpoint           = read_artifact("llm_endpoint")
        self.num_results            = int(read_artifact("num_results"))


        self.vsc   = VectorSearchClient(disable_notice=True)
        self.index = self.vsc.get_index(
            endpoint_name = self.vector_search_endpoint,
            index_name    = self.index_name
        )

    def retrieve_context(self, query: str, filters: Dict = None) -> List[Dict[str, Any]]:
        """Retrieve relevant facility documents using vector search."""
        search_kwargs = dict(
            query_text = query,
            columns    = [
                "id", "name", "document",
                "facility_type", "organization_type",
                "address_city", "address_stateOrRegion", "address_country",
                "numberDoctors", "capacity",
                "lat", "lon"
            ],
            num_results = self.num_results
        )

        if filters:
            search_kwargs["filters"] = filters

        results = self.index.similarity_search(**search_kwargs)

        documents = []
        if results and "result" in results and "data_array" in results["result"]:
            for row in results["result"]["data_array"]:
                documents.append({
                    "id"              : row[0],
                    "name"            : row[1],
                    "document"        : row[2],
                    "facility_type"   : row[3],
                    "organization_type": row[4],
                    "city"            : row[5],
                    "region"          : row[6],
                    "country"         : row[7],
                    "number_doctors"  : row[8],
                    "capacity"        : row[9],
                    "lat"             : row[10],
                    "lon"             : row[11],
                    "score"           : row[12]
                })

        return documents

    def format_context(self, docs: List[Dict]) -> str:
        """
        Build a clean context string from retrieved facility documents.
        Truncates each document to avoid exceeding LLM context window.
        """
        parts = []
        for i, doc in enumerate(docs, 1):
            part  = f"[Facility {i}]\n"
            part += f"Name: {doc['name']}\n"
            part += f"Type: {doc['facility_type']} ({doc['organization_type']})\n"
            part += f"Location: {doc['city']}, {doc['region']}, {doc['country']}\n"
            part += f"Doctors: {doc['number_doctors']} | Bed Capacity: {doc['capacity']}\n"
            part += f"Details:\n{doc['document'][:600]}\n"  # cap at 600 chars per facility
            parts.append(part)
        return "\n---\n".join(parts)

    def build_prompt(self, query: str, context_text: str) -> str:
        return f"""You are a medical facility assistant for the Virtue Foundation, helping NGO planners and healthcare coordinators find the right facilities.

Your job is to answer questions about healthcare facilities using ONLY the facility data provided below. Do not make up information or use outside knowledge.

Guidelines:
- Always mention the facility name and location (city, region) in your answer
- If a facility has unknown doctor count or capacity, mention that data is unavailable
- If multiple facilities match, list them clearly
- If no facility in the context matches the question, say "No matching facilities found in the provided data"
- Keep answers concise and structured

Facility Data:
{context_text}

Question: {query}

Answer:"""

    def generate_response(self, query: str, context_docs: List[Dict]) -> str:
        """Generate answer using LLM with retrieved facility context."""
        from mlflow.deployments import get_deploy_client

        context_text = self.format_context(context_docs)
        prompt       = self.build_prompt(query, context_text)

        deploy_client = get_deploy_client("databricks")
        response      = deploy_client.predict(
            endpoint = self.llm_endpoint,
            inputs   = {
                "messages": [
                    {"role": "system", "content": "You are a helpful medical facility assistant for Ghana. Answer only from provided context."},
                    {"role": "user",   "content": prompt}
                ],
                "max_tokens" : 600,
                "temperature": 0.0   # deterministic — factual queries need no creativity
            }
        )

        return response["choices"][0]["message"]["content"]

    def predict(self, context, model_input):
        # ── Parse input ──────────────────────────────────────────
        if isinstance(model_input, pd.DataFrame):
            questions = model_input["question"].tolist() \
                        if "question" in model_input.columns \
                        else model_input["query"].tolist()
            regions = model_input["region"].tolist() \
                    if "region" in model_input.columns \
                    else [None] * len(questions)
        elif isinstance(model_input, dict):
            questions = [model_input.get("question") or model_input.get("query", "")]
            regions   = [model_input.get("region", None)]
        else:
            questions = [str(model_input)]
            regions   = [None]

        results = []

        for query, region in zip(questions, regions):

            filters      = {"address_stateOrRegion": region} if region else None
            context_docs = self.retrieve_context(query, filters=filters)

            # ── No results ───────────────────────────────────────
            if not context_docs:
                results.append({
                    "question"          : query,
                    "answer"            : "No matching facilities found in the database.",
                    "citations"         : json.dumps([]),
                    "show_map_button"   : False,
                    "mappable_facilities": json.dumps([]),
                    "num_sources"       : 0
                })
                continue

            # ── Generate answer ──────────────────────────────────
            answer    = self.generate_response(query, context_docs)

            # ── Build citations ───────────────────────────────────
            citations = build_citations(context_docs)

            # ── Mappable facilities (have lat/lon) ────────────────
            mappable  = [
                {
                    "facility_id"  : c["facility_id"],
                    "name"         : c["name"],
                    "city"         : c["city"],
                    "region"       : c["region"],
                    "facility_type": c["facility_type"],
                    "lat"          : c["lat"],
                    "lon"          : c["lon"],
                    "rank"         : c["rank"]
                }
                for c in citations if c["has_location"]
            ]

            # ── Should frontend show map button? ──────────────────
            show_map  = is_facility_response(answer, citations)

            results.append({
                "question"           : query,
                "answer"             : answer,

                # All cited facilities (for reference/audit)
                "citations"          : json.dumps(citations),

                # Frontend uses this to show/hide the map button
                "show_map_button"    : show_map,

                # Frontend uses this to plot pins on the map
                "mappable_facilities": json.dumps(mappable),

                "num_sources"        : len(context_docs)
            })

        return pd.DataFrame(results)

In [0]:
import json
import tempfile
import os

# Create a test instance
test_model = FacilityRAGModel()

# Create temporary files for mock artifacts
artifacts = {}
tmpdir = tempfile.mkdtemp()

for key, value in {
    "vector_search_endpoint": VECTOR_SEARCH_ENDPOINT,
    "index_name"            : INDEX_NAME,
    "llm_endpoint"          : LLM_ENDPOINT,
    "num_results"           : str(NUM_RESULTS)
}.items():
    file_path = os.path.join(tmpdir, f"{key}.txt")
    with open(file_path, "w") as f:
        f.write(value)
    artifacts[key] = file_path

# Mock context for local testing
class MockContext:
    pass

mock_context = MockContext()
mock_context.artifacts = artifacts

test_model.load_context(mock_context)

# Test questions — cover different query types
test_inputs = pd.DataFrame({
    "question": [
        "What services does 2BN Military Hospital offer?",   # specific facility
        "Are there any clinics in Accra that's Internal Medicine?",     # capability search
    ],
    "region": [
        None,                   # no filter
        "Greater Accra Region", # with region filter
    ]
})

results = test_model.predict(None, test_inputs)

# ── Pretty print each result ─────────────────────────────────
for i, row in results.iterrows():
    print(f"\n{'='*60}")
    print(f"Question : {row['question']}")
    print(f"\nAnswer   : {row['answer']}")

    # Citations
    citations = json.loads(row["citations"])
    if citations:
        print(f"\nCitations ({len(citations)}):")
        for c in citations:
            location_flag = "(no GPS)" if not c["has_location"] else f"({c['lat']}, {c['lon']})"
            print(f"  [{c['rank']}] {c['name']} · {c['city']}, {c['region']} {location_flag} · score: {c['relevance_score']}")
    else:
        print("\nCitations: none")

    # Map button
    print(f"\nShow map button : {row['show_map_button']}")

    # Mappable facilities
    mappable = json.loads(row["mappable_facilities"])
    if mappable:
        print(f"Mappable facilities ({len(mappable)}):")
        for f in mappable:
            print(f"  - {f['name']} → lat: {f['lat']}, lon: {f['lon']}")
    else:
        print("Mappable facilities: none (missing GPS data)")

    print(f"\nTotal sources used: {row['num_sources']}")

In [0]:
import tempfile
import os
import mlflow
from mlflow.models.signature import infer_signature
import pandas as pd
from mlflow.models.resources import DatabricksVectorSearchIndex

# ============================================================
# Prepare artifacts
# ============================================================
artifacts = {}
tmpdir = tempfile.mkdtemp()  # persistent temp dir — not using context manager
                              # because MLflow reads artifacts AFTER the with block

for key, value in {
    "vector_search_endpoint": VECTOR_SEARCH_ENDPOINT,
    "index_name"            : INDEX_NAME,
    "llm_endpoint"          : LLM_ENDPOINT,
    "num_results"           : str(NUM_RESULTS)
}.items():
    file_path = os.path.join(tmpdir, f"{key}.txt")
    with open(file_path, "w") as f:
        f.write(value)
    artifacts[key] = file_path

print(f"Artifacts written to: {tmpdir}")

# ============================================================
# Input example — matches updated predict() signature
# includes both question and region columns
# ============================================================
input_example = pd.DataFrame({
    "question": [
        "What services does 2BN Military Hospital offer?",
        "Which hospitals have emergency care in Accra?"
    ],
    "region": [
        None,
        "Greater Accra Region"
    ]
})

# Get output from already-loaded test_model to infer signature
output_example = test_model.predict(None, input_example)

print("Output columns:", list(output_example.columns))
print("Output dtypes:\n", output_example.dtypes)

# ============================================================
# Infer signature from actual input/output
# ============================================================
from mlflow.models.signature import infer_signature

# Convert list/dict columns to string for signature inference
# (MLflow can't infer schema from JSON strings otherwise)
output_for_signature = output_example.copy()
output_for_signature["citations"]          = output_for_signature["citations"].astype(str)
output_for_signature["mappable_facilities"]= output_for_signature["mappable_facilities"].astype(str)
output_for_signature["show_map_button"]    = output_for_signature["show_map_button"].astype(bool)

signature = infer_signature(input_example, output_for_signature)
print("Signature inferred:")
print(signature)

# ============================================================
# Log model to MLflow
# ============================================================
current_user = spark.sql("SELECT current_user()").collect()[0][0]
experiment_path = f"/Users/{current_user}/facility-rag-chatbot"
mlflow.set_experiment(experiment_path)

resources = [
    DatabricksVectorSearchIndex(
        index_name="workspace.default.facility_documents_index"
    )
]

with mlflow.start_run(run_name="facility_rag_v1") as run:

    mlflow.pyfunc.log_model(
        artifact_path = "model",
        python_model  = FacilityRAGModel(),
        resources=resources,
        artifacts     = artifacts,
        signature     = signature,
        input_example = input_example,
        pip_requirements = [
            "mlflow",
            "databricks-vectorsearch",
        ]
    )

    # Log config as params for traceability in MLflow UI
    mlflow.log_params({
        "vector_search_endpoint": VECTOR_SEARCH_ENDPOINT,
        "index_name"            : INDEX_NAME,
        "llm_endpoint"          : LLM_ENDPOINT,
        "num_results"           : NUM_RESULTS,
        "model_version"         : "v1"
    })

    # Log a quick eval metric
    mlflow.log_metric("num_facilities_indexed", 
                      spark.table(INDEX_NAME.split(".")[-1] 
                                  if "." in INDEX_NAME 
                                  else INDEX_NAME).count()
                      if False else 750)   # replace 750 with actual count

    run_id   = run.info.run_id
    model_uri = f"runs:/{run_id}/model"

    print(f"Experiment  : {experiment_path}")
    print(f"Run ID      : {run_id}")
    print(f"Model URI   : {model_uri}")

In [0]:
# Register model to Unity Catalog
uc_model_name = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=uc_model_name
)

print(f"\n✓ Model registered to Unity Catalog!")
print(f"Model name: {uc_model_name}")
print(f"Version: {registered_model.version}")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput
from databricks.sdk.service.serving import EndpointStateReady
import time

w = WorkspaceClient()

existing_endpoints = [e.name for e in w.serving_endpoints.list()]

if SERVING_ENDPOINT_NAME in existing_endpoints:
    print(f"Updating existing endpoint '{SERVING_ENDPOINT_NAME}'...")
    w.serving_endpoints.update_config(
        name           = SERVING_ENDPOINT_NAME,
        served_entities = [
            ServedEntityInput(
                entity_name          = uc_model_name,
                entity_version       = str(registered_model.version),
                scale_to_zero_enabled= True,
                workload_size        = "Small"
            )
        ]
    )
    print("Update triggered — waiting for endpoint to be ready...")

else:
    print(f"Creating new endpoint '{SERVING_ENDPOINT_NAME}'...")
    w.serving_endpoints.create(
        name   = SERVING_ENDPOINT_NAME,
        config = EndpointCoreConfigInput(
            served_entities = [
                ServedEntityInput(
                    entity_name          = uc_model_name,
                    entity_version       = str(registered_model.version),
                    scale_to_zero_enabled= True,
                    workload_size        = "Small"
                )
            ]
        )
    )
    print("Creation triggered — waiting for endpoint to be ready...")

# ============================================================
# Poll until endpoint is ready
# ============================================================
while True:
    state = w.serving_endpoints.get(SERVING_ENDPOINT_NAME).state
    ready = state.ready if state else None
    print(f"State: {ready}")

    if ready == EndpointStateReady.READY:
        print("Endpoint is ready!")
        break
    elif ready == EndpointStateReady.NOT_READY:
        print("Still provisioning... waiting 30s")
        time.sleep(30)
    else:
        print(f"Unexpected state: {ready} — check Databricks UI > Serving")
        break

# ============================================================
# Print final details
# ============================================================
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")

print(f"\nEndpoint name : {SERVING_ENDPOINT_NAME}")
print(f"Serving model : {uc_model_name} v{registered_model.version}")
print(f"Endpoint URL  : https://{workspace_url}/serving-endpoints/{SERVING_ENDPOINT_NAME}")
print(f"Invocation URL: https://{workspace_url}/serving-endpoints/{SERVING_ENDPOINT_NAME}/invocations")

In [0]:
import time
import requests
import json
from databricks.sdk.service.serving import EndpointStateReady

# ============================================================
# Endpoint is already ready from previous cell — skip the wait
# Just do a quick state check before testing
# ============================================================
state = w.serving_endpoints.get(SERVING_ENDPOINT_NAME).state.ready
if state == EndpointStateReady.READY:
    print(f"Endpoint '{SERVING_ENDPOINT_NAME}' is ready — proceeding to test")
else:
    print(f"Endpoint state: {state} — may not respond correctly")

# ============================================================
# Test payload — updated to include region column
# (matches the new predict() signature)
# ============================================================
test_data = {
    "dataframe_records": [
        {
            "question": "What services does 2BN Military Hospital offer?",
            "region"  : None
        },
        {
            "question": "Are there any facilities with emergency care in Accra?",
            "region"  : "Greater Accra Region"
        }
    ]
}

# ============================================================
# Auth + request
# ============================================================
token         = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
endpoint_url  = f"https://{workspace_url}/serving-endpoints/{SERVING_ENDPOINT_NAME}/invocations"

print(f"\nCalling: {endpoint_url}\n")

response = requests.post(
    endpoint_url,
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type" : "application/json"
    },
    json    = test_data,
    timeout = 120   # RAG calls take longer than simple models
)

# ============================================================
# Parse and print results
# ============================================================
if response.status_code == 200:
    results = response.json()
    print("Endpoint test successful!\n")

    predictions = results.get("predictions", [])

    for i, pred in enumerate(predictions):
        print(f"{'='*60}")
        print(f"Question      : {pred['question']}")
        print(f"\nAnswer        : {pred['answer']}")
        print(f"\nMap button    : {pred['show_map_button']}")
        print(f"Num sources   : {pred['num_sources']}")

        # Citations
        citations = json.loads(pred["citations"]) if isinstance(pred["citations"], str) else pred["citations"]
        if citations:
            print(f"\nCitations ({len(citations)}):")
            for c in citations:
                loc = f"lat: {c['lat']}, lon: {c['lon']}" if c["has_location"] else "no GPS"
                print(f"  [{c['rank']}] {c['name']} · {c['city']}, {c['region']} · {loc} · score: {c['relevance_score']}")
        else:
            print("\nCitations     : none")

        # Mappable facilities
        mappable = json.loads(pred["mappable_facilities"]) if isinstance(pred["mappable_facilities"], str) else pred["mappable_facilities"]
        if mappable:
            print(f"\nMappable facilities ({len(mappable)}):")
            for f in mappable:
                print(f"  - {f['name']} → ({f['lat']}, {f['lon']})")
        else:
            print("\nMappable      : none (missing GPS data)")

else:
    print(f"Error {response.status_code}:")
    print(response.text)